# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item (or order-line combination).

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
from IPython.display import display

lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'],
)

assert len(lines) == 4
assert lines['qty'].sum() == 7

display(lines)
print(f"{len(lines)} line items; {lines['qty'].sum()} total units.")

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


4 line items; 7 total units.


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [4]:
vendors = pd.json_normalize(vendors_json)
joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True,
    validate='many_to_one',
)

unmatched = joined.loc[joined['_merge'].eq('left_only')]
unmatched_units = (
    unmatched.groupby('vendor_id', as_index=False)['qty']
    .sum()
    .rename(columns={'qty': 'total_qty'})
)

display(joined)
print('Unmatched vendors and units:')
display(unmatched_units)
# V-07 did not match; its Hot Dog line accounts for 3 units.

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


Unmatched vendors and units:


,vendor_id,total_qty
0,V-07,3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [6]:
# Label unmatched vendors explicitly, including their ID, so their units are
# retained and different, unknown vendors don't collapse into one unnamed group
vendor_names = joined['name'].fillna('Unknown vendor (' + joined['vendor_id'] + ')')
vendor_totals = (
    joined.assign(vendor_name=vendor_names)
    .groupby('vendor_name', as_index=False)['qty']
    .sum()
    .rename(columns={'qty': 'total_qty'})
    .sort_values(['total_qty', 'vendor_name'], ascending=[False, True])
    .reset_index(drop=True)
)

display(vendor_totals)
assert vendor_totals['total_qty'].sum() == 7

,vendor_name,total_qty
0,Hoos Burgers,3
1,Unknown vendor (V-07),3
2,Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [8]:
# Compare only known zones and account for the missing-zone units separately.
zone_totals = (
    joined.dropna(subset=['zone'])
    .groupby('zone', as_index=False)['qty']
    .sum()
    .rename(columns={'qty': 'total_qty'})
    .sort_values('total_qty', ascending=False)
    .reset_index(drop=True)
)
known_zone_units = zone_totals['total_qty'].sum()
unknown_zone_units = joined.loc[joined['zone'].isna(), 'qty'].sum()
best_zones = zone_totals.loc[
    zone_totals['total_qty'].eq(zone_totals['total_qty'].max()), 'zone'
].tolist()

display(zone_totals)
print(f"Top known zone(s): {', '.join(best_zones)} "
      f"with {zone_totals['total_qty'].max()} units each.")
print(f"Known-zone total: {known_zone_units} of {joined['qty'].sum()} units.")
print(f"The remaining {unknown_zone_units} units belong to unmatched vendor V-07, "
      'whose zone is unknown.')
# Zone A has 4 units, notably the most among known zones. We cannot assign V-07's
# 3 units to a zone without additional vendor information.
assert known_zone_units + unknown_zone_units == joined['qty'].sum()

,zone,total_qty
0,A,4


Top known zone(s): A with 4 units each.
Known-zone total: 4 of 7 units.
The remaining 3 units belong to unmatched vendor V-07, whose zone is unknown.


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [9]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# Supply empty list for absent lines without changing the original payload
# A missing lines key gives us no line items to flatten, not a zero-quantity row.
prepared_orders = [{**order, 'lines': order.get('lines', [])} for order in ragged_json]
orders_without_lines = [order['order'] for order in ragged_json if 'lines' not in order]
ragged_lines = pd.json_normalize(
    prepared_orders,
    record_path='lines',
    meta=['order', 'vendor_id'],
)
# Nullable integers preserve the unknown quantity as <NA>; a real zero stays 0.
ragged_lines['qty'] = ragged_lines['qty'].astype('Int64')

display(ragged_lines)
print(f"Orders with no lines key: {orders_without_lines}; no line-item rows emitted.")
print(f"Line items with unknown quantity: {ragged_lines['qty'].isna().sum()}.")
print('The observed quantity subtotal is 2; the full total is unknown because '
      'the Fries quantity is missing.')

,item,qty,order,vendor_id
0,Soda,2,4,V-01
1,Fries,<NA>,6,V-01


Orders with no lines key: [5]; no line-item rows emitted.
Line items with unknown quantity: 1.
The observed quantity subtotal is 2; the full total is unknown because the Fries quantity is missing.


### Q6 — Validate

In [11]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** One row per line item makes it easier to work with items and their quantities while maintaining the ability to know what order/vendor they belong to. For example, we can add up how many units each vendor sold with a group-by and sum. If we kept one row per order, we would have to pull the items out of the nested lists first. Also, counting rows would only tell us how many line items there are and not exactly how many units were sold.

**b)** A missing quantity means we simply don't know the amount, while zero means we do know the amount, and that amount is zero. If we replace missing quantities with zero, we make the data look complete even though some information isn't there. This would throw off totals, averages, and other important inventory-related calculations. In Q5, we know there are 2 Sodas, but we don't know how many Fries there are, so we can't say the full total is 2. Pandas skip missing values when adding by default, so we still need to check for these values. Order 5 has no lines listed, so it doesn't produce a line-item row, but that doesn't necessarily mean it had zero units.